# 2 · Entrenar el LoRA de Lía (SDXL): cara y cuerpo
Antes: elige las **20–30 imágenes** de `lia2/candidatas` donde cara y cuerpo se parezcan (mezcla retratos, cintura para arriba y cuerpo
entero) y pon sus números en la lista `ELEGIDAS` de la primera celda: se copian solas a `lia2/dataset`.
**GPU T4.** Tiempo aproximado: 1 h (estimado, no medido). El resultado queda en `lia2/lora`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/lia2'      # todo se guarda aquí, en tu Google Drive
for d in ('candidatas', 'dataset', 'lora'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA -> Entorno de ejecución > Cambiar tipo > GPU T4')

In [ ]:
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0 transformers accelerate peft safetensors

In [ ]:
# Números de img_XX.png elegidos de lia2/candidatas (p. ej. [1, 4, 7, ...]). Se copian solos a lia2/dataset.
ELEGIDAS = []
import shutil
for n in ELEGIDAS:
    shutil.copy(f'{BASE}/candidatas/img_{n:02d}.png', f'{BASE}/dataset/')
print(len(ELEGIDAS), 'fotos copiadas a lia2/dataset')

In [ ]:
!pip install -q bitsandbytes
!wget -q -O train_dreambooth_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/v0.41.0/examples/dreambooth/train_dreambooth_lora_sdxl.py
import glob
n = len(glob.glob(f'{BASE}/dataset/*.png') + glob.glob(f'{BASE}/dataset/*.jpg'))
print(n, 'imágenes en el dataset')
assert n >= 10, 'Pon en ELEGIDAS los números de tus mejores imágenes (mínimo 10, ideal 20-30)'

In [ ]:
!accelerate launch --mixed_precision=fp16 train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path=SG161222/RealVisXL_V4.0 \
  --pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix \
  --instance_data_dir={BASE}/dataset \
  --instance_prompt="photo of lia_character woman" \
  --output_dir={BASE}/lora \
  --resolution=768 --center_crop \
  --train_batch_size=1 --gradient_accumulation_steps=2 \
  --gradient_checkpointing --use_8bit_adam --mixed_precision=fp16 \
  --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 \
  --rank=16 --max_train_steps=1000 --checkpointing_steps=500 --seed=42

In [ ]:
import os
f = f'{BASE}/lora/pytorch_lora_weights.safetensors'
print('OK, LoRA guardado:' if os.path.exists(f) else 'NO se creó el LoRA, revisa los errores de arriba:', f)